# Problem 5: Retention, RFM and the first delivery

97% of customers only ever place one order, so growth depends almost completely on new customers. In this notebook I look at:
- how retention looks by monthly cohort
- which customers are worth a win-back campaign (RFM)
- whether a late first delivery makes customers less likely to order again

Note: I use `customer_unique_id` for customers, because `customer_id` is different for every order.

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
import chart_style as cs
cs.apply(); warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:,.2f}".format)
con = sqlite3.connect(ROOT / "data" / "olist.db")
OUT = ROOT / "data" / "processed"

from scipy import stats

In [2]:
co = pd.read_csv(OUT / "customer_orders.csv", parse_dates=["order_purchase_timestamp"])
co = co[co.purchase_month.between("2017-01", "2018-08")]
rep = pd.read_csv(OUT / "repeat_distribution.csv")
print(f"Customers: {co.customer_unique_id.nunique():,} | delivered orders: {len(co):,}")
rep

Customers: 93,096 | delivered orders: 96,203


,orders_per_customer,customers,pct_customers
0,1,90557,97.00
1,2,2573,2.76
2,3,181,0.19
3,4+,47,0.05


## 1. Cohort retention

In [3]:
first = co.groupby("customer_unique_id").order_purchase_timestamp.min().dt.to_period("M").rename("cohort")
c2 = co.join(first, on="customer_unique_id")
c2["period"] = (c2.order_purchase_timestamp.dt.to_period("M") - c2.cohort).apply(lambda x: x.n)
cohort = (c2.groupby(["cohort", "period"]).customer_unique_id.nunique().unstack(fill_value=0))
size = cohort[0]
ret = cohort.div(size, axis=0) * 100
ret = ret.loc[:"2018-02", 1:12]          # cohorts with >= 6 months of follow-up
for c in ret.index:                       # hide months not yet observed at the data cut-off
    for k in ret.columns:
        if c + int(k) > pd.Period("2018-08", "M"): ret.loc[c, k] = np.nan
cum = ret.cumsum(axis=1, skipna=False)

fig, ax = plt.subplots(figsize=(11, 5.2))
cmap = __import__("matplotlib.colors", fromlist=["x"]).LinearSegmentedColormap.from_list("b", ["#f7fcb9", "#d9f0a3", "#addd8e", "#78c679", "#41ab5d", "#238443", "#005a32"])
cmap.set_bad("white")
im = ax.imshow(np.ma.masked_invalid(ret.values.astype(float)), cmap=cmap, aspect="auto", vmin=0, vmax=0.8)
for i in range(ret.shape[0]):
    for j in range(ret.shape[1]):
        v = ret.values[i, j]
        if not np.isnan(v) and (ret.index[i] + int(ret.columns[j])) <= pd.Period("2018-08", "M"):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7.5, color="white" if v > 0.45 else cs.DARK)
ax.set_xticks(range(ret.shape[1]), [f"M+{c}" for c in ret.columns]); ax.set_yticks(range(ret.shape[0]), [f"{p} (n={size[p]:,})" for p in ret.index], fontsize=8)
ax.grid(False); ax.set_xlabel("Months after first purchase")
cb = fig.colorbar(im, ax=ax, shrink=0.8); cb.set_label("% of cohort buying again in that month", color=cs.GREY); cb.outline.set_visible(False)
ax.set_title("Monthly cohort retention (% of cohort ordering again)")
cs.save(fig, "p5_cohort_retention")
cum[[1, 3, 6]].mean().rename("avg cumulative repeat % (M+1, M+3, M+6)").round(2)

period
1   0.47
3   1.07
6   1.89
Name: avg cumulative repeat % (M+1, M+3, M+6), dtype: float64

## 2. RFM segments

Normal RFM quintiles don't work well here because almost everyone has frequency = 1. So I use quintiles for Recency and Monetary, and just a yes/no flag for repeat buyers. Snapshot date is 2018-09-01.

In [4]:
snap = pd.Timestamp("2018-09-01")
rfm = co.groupby("customer_unique_id").agg(last=("order_purchase_timestamp", "max"),
                                           frequency=("order_id", "count"), monetary=("paid_value", "sum"))
rfm["recency_days"] = (snap - rfm["last"]).dt.days
rfm["R"] = pd.qcut(rfm.recency_days, 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["M"] = pd.qcut(rfm.monetary.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment(r_):
    if r_.frequency >= 2 and r_.R >= 3: return "Loyal repeaters"
    if r_.frequency >= 2:               return "Lapsed repeaters"
    if r_.R >= 4 and r_.M >= 4:         return "New high-value"
    if r_.R >= 4:                       return "New low-value"
    if r_.M >= 4:                       return "At-risk high-value"
    return "Dormant low-value"
rfm["segment"] = rfm.apply(segment, axis=1)
seg = (rfm.groupby("segment").agg(customers=("monetary", "size"), revenue=("monetary", "sum"),
                                  avg_spend=("monetary", "mean"), avg_recency=("recency_days", "mean")))
seg["pct_customers"] = seg.customers / seg.customers.sum() * 100
seg["pct_revenue"] = seg.revenue / seg.revenue.sum() * 100
order = ["Loyal repeaters", "Lapsed repeaters", "New high-value", "At-risk high-value", "New low-value", "Dormant low-value"]
seg = seg.loc[order]
rfm.reset_index()[["customer_unique_id", "recency_days", "frequency", "monetary", "R", "M", "segment"]].to_csv(OUT / "customer_rfm.csv", index=False)
seg.round(1)

,customers,revenue,avg_spend,avg_recency,pct_customers,pct_revenue
segment,,,,,,
Loyal repeaters,1799,"568,148.30",315.80,132.80,1.90,3.70
Lapsed repeaters,990,"292,606.30",295.60,381.60,1.10,1.90
New high-value,14401,"4,350,459.20",302.10,92.80,15.50,28.30
At-risk high-value,20610,"6,146,699.00",298.20,335.30,22.10,40.00
New low-value,21715,"1,583,127.90",72.90,91.30,23.30,10.30
Dormant low-value,33581,"2,433,455.80",72.50,337.50,36.10,15.80


In [5]:
fig, ax = plt.subplots(figsize=(10, 4.2))
y = np.arange(len(seg)); h = 0.38
ax.barh(y - h/2 - 0.01, seg.pct_customers, h, color=cs.GREEN, label="% of customers")
ax.barh(y + h/2 + 0.01, seg.pct_revenue, h, color=cs.ORANGE, label="% of revenue")
for i, (a, b) in enumerate(zip(seg.pct_customers, seg.pct_revenue)):
    ax.text(a + 0.4, i - h/2, f"{a:.1f}%", va="center", fontsize=8.5, color=cs.GREY)
    ax.text(b + 0.4, i + h/2, f"{b:.1f}%", va="center", fontsize=8.5, color=cs.GREY)
ax.set_yticks(y, seg.index); ax.invert_yaxis(); ax.grid(axis="y", visible=False)
ax.set_xlabel("% of total"); ax.legend(loc="lower right")
ax.set_title("RFM segments: share of customers vs share of revenue")
hv = seg.loc[["New high-value", "At-risk high-value"]]
cs.save(fig, "p5_rfm_segments")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p5_rfm_segments.png')

## 3. Late first delivery vs repeat purchase

I only use customers whose first order was before March 2018, so everyone had at least 6 months to come back.

In [6]:
f = co[co.order_seq == 1].copy()
nxt = co.groupby("customer_unique_id").order_id.count().rename("n_orders")
f = f.join(nxt, on="customer_unique_id")
f = f[f.order_purchase_timestamp < "2018-03-01"].copy()
f["repeat"] = (f.n_orders > 1).astype(int)
tab = f.groupby("is_late").repeat.agg(["count", "sum", "mean"]).rename(index={0: "On-time first order", 1: "Late first order"})
z, p = __import__("statsmodels.stats.proportion", fromlist=["x"]).proportions_ztest(tab["sum"].values, tab["count"].values)
rel = tab.loc["Late first order", "mean"] / tab.loc["On-time first order", "mean"] - 1
print(f"Repeat rate on time {tab.iloc[0]['mean']:.2%} vs late {tab.iloc[1]['mean']:.2%} | relative change {rel:+.1%} | z = {z:.2f}, p = {p:.4f}")
tab

Repeat rate on time 4.04% vs late 3.19% | relative change -21.0% | z = 2.72, p = 0.0065


,count,sum,mean
is_late,,,
On-time first order,50997,2060,0.04
Late first order,4262,136,0.03


In [7]:
# Control for spend, state and purchase month: lateness clusters in certain regions and months
f["log_spend"] = np.log1p(f.paid_value)
f["month"] = f.purchase_month
top_states = f.customer_state.value_counts().index[:10]
f["state"] = np.where(f.customer_state.isin(top_states), f.customer_state, "other")
m1 = smf.logit("repeat ~ is_late + log_spend + C(state) + C(month)", data=f).fit(disp=False)
or_late = np.exp(m1.params["is_late"]); ci = np.exp(m1.conf_int().loc["is_late"])
print(f"Adjusted odds ratio of repeating after a late first order: {or_late:.2f} (95% CI {ci[0]:.2f}-{ci[1]:.2f}), p = {m1.pvalues['is_late']:.4f}")

Adjusted odds ratio of repeating after a late first order: 0.87 (95% CI 0.73-1.04), p = 0.1342


In [8]:
by_score = f.groupby("review_score").repeat.agg(["count", "mean"]).assign(mean=lambda t: t["mean"] * 100)
fig, axes = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={"wspace": 0.35, "width_ratios": [1, 1.4]})
ax = axes[0]
vals = tab["mean"] * 100
ax.bar(["On-time\nfirst order", "Late\nfirst order"], vals, color=[cs.GREEN, cs.ORANGE], width=0.55)
for i, v in enumerate(vals): ax.text(i, v + 0.06, f"{v:.2f}%", ha="center", fontsize=10, color=cs.DARK)
ax.set_ylabel("% who bought again"); ax.set_ylim(0, 5); ax.grid(axis="x", visible=False)
ax.set_title("Repeat rate: on-time vs late first order")
ax = axes[1]
ax.bar(by_score.index.astype(int).astype(str) + "-star", by_score["mean"], color=cs.PLUM, width=0.55)
for i, v in enumerate(by_score["mean"]): ax.text(i, v + 0.06, f"{v:.2f}%", ha="center", fontsize=9, color=cs.GREY)
ax.set_ylim(0, 5); ax.set_xlabel("Review score of the first order"); ax.grid(axis="x", visible=False)
ax.set_title("Repeat rate by first-order review score")
cs.save(fig, "p5_first_order_effect")

PosixPath('/home/claude/olist-ops-growth-case-study/images/p5_first_order_effect.png')

The raw difference (4.04% vs 3.19%) is significant. But late deliveries happen more in some months (Nov 2017, Feb-Mar 2018) and some states, and those also have different repeat rates. After controlling for spend, state and month the odds ratio is 0.87 with p = 0.13, so it's not significant anymore. My conclusion is that late delivery clearly hurts the review, but I can't say from this data that it reduces repeat purchases. Low retention seems to be a general problem, not mainly a delivery problem.

## 4. Value in BRL (upper bound)

Using the raw gap gives the maximum possible value.

In [9]:
annual_first_orders = co[(co.order_seq == 1) & co.purchase_month.between("2017-09", "2018-08")].shape[0]
late_share = f.is_late.mean()
gap = tab.iloc[0]["mean"] - tab.iloc[1]["mean"]
repeat_value = co[co.order_seq >= 2].groupby("customer_unique_id").paid_value.sum().mean()
lost_customers = annual_first_orders * late_share * gap
val = pd.Series({
    "First-time customers / year (Sep-17 to Aug-18)": annual_first_orders,
    "Share whose first order is late": late_share,
    "Repeat-rate gap (on time − late)": gap,
    "Repeat customers lost per year": lost_customers,
    "Avg later-order value of a repeat customer (BRL)": repeat_value,
    "Repeat revenue lost per year (BRL)": lost_customers * repeat_value})
pd.DataFrame({"metric": val.index, "value": val.values}).to_csv(OUT / "retention_value.csv", index=False)
val.round(4)

First-time customers / year (Sep-17 to Aug-18)     71,686.00
Share whose first order is late                         0.08
Repeat-rate gap (on time − late)                        0.01
Repeat customers lost per year                         46.91
Avg later-order value of a repeat customer (BRL)      162.59
Repeat revenue lost per year (BRL)                  7,627.42
dtype: float64

## Summary and recommendations

| Finding | What I'd do | KPI |
|---|---|---|
| Only ~3% of customers order again; month-1 return rate is under 1% | After delivery: ask for a review, then send an offer from the same category around day 30 | 90-day repeat rate |
| "New high-value" and "At-risk high-value" customers bring a big share of revenue but ordered only once | Run win-back campaigns only for these two groups, skip low-value dormant customers | Win-back conversion, ROI |
| Late first order: -21% repeat rate in raw data, but not significant after controls (OR 0.87, p = 0.13) | Don't use retention as the argument for logistics fixes; if needed, A/B test a delay voucher | Repeat rate in the test |
| Even the upper bound of lost repeat revenue is small (~R$ 8k/year), while 54% of late orders get 1-2 stars | Justify the delivery work with reviews; improve retention through CRM | Avg review, 1-2 star share, 90-day repeat |